In [1]:
import re
from datetime import datetime
from typing import Dict, Any
import uuid
import json

class ClinicalDocumentParser:
    def __init__(self):
        # Heuristics based on the messy OCR structure you provided
        self.doc_divider_pattern = re.compile(
            r"(CLINICAL PATHOLOGY REPORT|NURSES NOTES|CONSULTATION SHEET|DRUG CHART|ADMISSION RECORD|DISCHARGE CHECK LIST|ER OBSERVATION CHART)", 
            re.IGNORECASE
        )
        # Patterns for deterministic extraction
        self.dob_pattern = re.compile(r"(?:Date of Birth|DOB)[\s:]*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})", re.IGNORECASE)
        self.admit_date_pattern = re.compile(r"Date Of Admission[\s:]*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})", re.IGNORECASE)
        self.gender_pattern = re.compile(r"Gender[\s:]*(Male|Female|M|F)", re.IGNORECASE)

    def _calculate_age(self, dob_str: str, admit_str: str) -> int:
        try:
            dob = datetime.strptime(dob_str.replace('-', '/'), "%d/%m/%Y")
            admit_date = datetime.strptime(admit_str.replace('-', '/'), "%d/%m/%Y")
            return admit_date.year - dob.year - ((admit_date.month, admit_date.day) < (dob.month, dob.day))
        except ValueError:
            return -1 

    def parse_raw_text(self, patient_id: str, raw_text: str) -> Dict[str, Any]:
        dob_match = self.dob_pattern.search(raw_text)
        admit_match = self.admit_date_pattern.search(raw_text)
        gender_match = self.gender_pattern.search(raw_text)

        age = -1
        if dob_match and admit_match:
            age = self._calculate_age(dob_match.group(1), admit_match.group(1))
            
        gender = gender_match.group(1).capitalize() if gender_match else "Unknown"

        chunks = self.doc_divider_pattern.split(raw_text)
        documents = []
        
        if chunks[0].strip():
            documents.append({
                "doc_id": str(uuid.uuid4()),
                "doc_type": "GENERAL_ADMISSION",
                "timestamp": admit_match.group(1) if admit_match else "Unknown",
                "raw_content": chunks[0].strip()[:500] + "..." # Truncated for notebook display
            })
            
        for i in range(1, len(chunks), 2):
            doc_type = chunks[i].strip().upper()
            content = chunks[i+1].strip() if i+1 < len(chunks) else ""
            
            if content:
                documents.append({
                    "doc_id": str(uuid.uuid4()),
                    "doc_type": doc_type,
                    "timestamp": admit_match.group(1) if admit_match else "Unknown",
                    "raw_content": content.strip()[:500] + "..." # Truncated for notebook display
                })

        return {
            "patient_id": patient_id,
            "preprocessed_age": age,
            "preprocessed_gender": gender,
            "chunk_count": len(documents),
            "chronological_docs": documents
        }

# --- TEST EXECUTION ---

# Snippet of the raw OCR data from Patient 2
sample_ocr_text = """
DIAGNOSIS:
1) ACUTE GASTROENTERITIS WITH DEHYDRATION
2) URINARY TRACT INFECTION
HISTORY: C/O Multiple episodes of loose stools, 2-3 episodes of vomiting,
 fatigue since 3 days and fever since yesterday.

NURSING DOCUMENTATION
(From.....
8:15
spm 8
 .to....
SAM
Date: 27/2/26
Patient hand overtaken from
9pm

DISCHARGE CHECK LIST
The following table:
"SL NO","DETAILS","REMARKS"
"1","PATIENT DETAILS FULL NAME AGE/GENDER IP NO DATE OF ADMISSION DATE OF DISCHARGE"
"""

# Initialize parser and run
parser = ClinicalDocumentParser()
parsed_data = parser.parse_raw_text(patient_id="PATIENT_002", raw_text=sample_ocr_text)

# Pretty print the output
print(json.dumps(parsed_data, indent=2))

{
  "patient_id": "PATIENT_002",
  "preprocessed_age": -1,
  "preprocessed_gender": "Unknown",
  "chunk_count": 2,
  "chronological_docs": [
    {
      "doc_id": "9c8c23be-244b-4621-8c2d-c1657be76f42",
      "doc_type": "GENERAL_ADMISSION",
      "timestamp": "Unknown",
      "raw_content": "DIAGNOSIS:\n1) ACUTE GASTROENTERITIS WITH DEHYDRATION\n2) URINARY TRACT INFECTION\nHISTORY: C/O Multiple episodes of loose stools, 2-3 episodes of vomiting,\n fatigue since 3 days and fever since yesterday.\n\nNURSING DOCUMENTATION\n(From.....\n8:15\nspm 8\n .to....\nSAM\nDate: 27/2/26\nPatient hand overtaken from\n9pm..."
    },
    {
      "doc_id": "cf8351ae-d86d-4795-b060-21191416fb37",
      "doc_type": "DISCHARGE CHECK LIST",
      "timestamp": "Unknown",
      "raw_content": "The following table:\n\"SL NO\",\"DETAILS\",\"REMARKS\"\n\"1\",\"PATIENT DETAILS FULL NAME AGE/GENDER IP NO DATE OF ADMISSION DATE OF DISCHARGE\"..."
    }
  ]
}
